# Learning Stable Control-oriented Deep Koopman Operators  


This tutorial demonstrates the use of [Deep Koopman Operators](https://www.nature.com/articles/s41467-018-07210-0) (DeepKO) with stability guarantees for system identificaiton of nonautonomous nonlinear dynamical systems in NeuroMANCER. 


## Koopman System Identification with Control Problem Setup

<img src="../figs/Koopman_model.png" width="500">  


System observables  $y_{k}$ and control inputs $u_k$ are encoded via encoder neural networks $f_y$ and $f_u$ to generate latent states $x_{k}$ (linear coordinates). This coordinate transformation now allos to apply linear Koopman operator $K$ to obtain latent states at the next time step $x_{k+1}$. After the rollout over given prediction horizon $N$ the generated latent trajectories $\{x_1, ..., x_N \}$ are proejcted back to the observable space via decoder neural network $f_y^{-1}$.
Now this Koopman encoder-decoder model can be trained as any other black-box nonlinear system identification problem with the loss $\mathcal{L}$ evaluated by comparing the  trajectory generated by the model with the training trajectory. 


### Koopman with Control References

[1] [H. Shi and M. Q. . -H. Meng, "Deep Koopman Operator With Control for Nonlinear Systems," in IEEE Robotics and Automation Letters, vol. 7, no. 3, pp. 7700-7707, July 2022, doi: 10.1109/LRA.2022.3184036.](https://ieeexplore.ieee.org/document/9799788)  
[2] [Eurika Kaiser and J Nathan Kutz and Steven L Brunton, Data-driven discovery of Koopman eigenfunctions for control, Mach. Learn.: Sci. Technol. 2021](https://iopscience.iop.org/article/10.1088/2632-2153/abf0f5)  
[3] [M. Korda and I. Mezić, "Optimal Construction of Koopman Eigenfunctions for Prediction and Control," in IEEE Transactions on Automatic Control, vol. 65, no. 12, pp. 5114-5129, Dec. 2020](https://ieeexplore.ieee.org/document/9022864)  
[4] [Yiqiang Han, Wenjian Hao, Umesh Vaidya, Deep Learning of Koopman Representation for Control, 2020
](https://arxiv.org/abs/2010.07546)  
[6] [Minghao Han, Jacob Euler-Rolle, Robert K. Katzschmann, DeSKO: Stability-Assured Robust Control with a Deep Stochastic Koopman Operator, ICLR 2022](https://openreview.net/forum?id=hniLRD_XCA)  
[6] https://github.com/HaojieSHI98/DeepKoopmanWithControl


### Generic Stable Layers References
[7]  [E. Skomski, S. Vasisht, C. Wight, A. Tuor, J. Drgoňa and D. Vrabie, "Constrained Block Nonlinear Neural Dynamical Models," 2021 American Control Conference (ACC), New Orleans, LA, USA, 2021, pp. 3993-4000](https://ieeexplore.ieee.org/document/9482930)   
[8] [J. Drgoňa, A. Tuor, S. Vasisht and D. Vrabie, "Dissipative Deep Neural Dynamical Systems," in IEEE Open Journal of Control Systems, 2022](https://ieeexplore.ieee.org/abstract/document/9809789)  
[9] [Jiong Zhang, Qi Lei, Inderjit S. Dhillon, Stabilizing Gradients for Deep Neural Networks via Efficient SVD Parameterization, InternationalConferenceonMachine Learning, 2018](https://arxiv.org/abs/1803.09327)



## NeuroMANCER and Dependencies

### Install (Colab only)
Skip this step when running locally.

In [114]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pickle 
import scipy
import time as tim
from scipy.io import loadmat

from neuromancer.psl import plot
from neuromancer import psl
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from neuromancer.system import Node, System
from neuromancer.slim import slim
from neuromancer.trainer import Trainer
from neuromancer.problem import Problem
from neuromancer.dataset import DictDataset
from neuromancer.constraint import variable
from neuromancer.loss import PenaltyLoss
from neuromancer. modules import blocks
import joblib

from sklearn.preprocessing import StandardScaler
torch.manual_seed(0)

## Generate trajectories from ODE system 

In this example we don't assume any prior knowledge on the system dynamics. We will only have access to limited measurements of the system states $x$ of an unknown [ordinary differential equations](https://en.wikipedia.org/wiki/Ordinary_differential_equation) (ODE).

Select the system_name from the available list of [nonautonomous ODE systems](https://github.com/pnnl/neuromancer/blob/master/src/neuromancer/psl/nonautonomous.py):
- TwoTank
- CSTR
- SwingEquation
- IverSimple

# Get and save data - (Already done)

## Create training data of sampled trajectories

We will obtain a dataset of sampled trajectories of the system dynamics in the form of input-state tuples: 
$$D = \big[(u^i_0, \hat{x}^i_0), ..., (u^i_N, \hat{x}^i_{N}) \big], \, \, i \in [1, ..., m]$$
where $N$ represents the prediction horizon, $m$ represents number of measured trajectories, and $i$ represents an index of the sampled trajectory.
Variables $x_k$ represent system states, and $u_k$ are exogenous inputs at time $k$.

In [115]:
# Load data from .mat files
train_data = loadmat('train_data_ident.mat')
test_data = loadmat('test_data_ident.mat')

# Extract variables
Y_train = train_data['Ytrain']
U_train = train_data['Utrain']
Y_test = test_data['Ytest']
U_test = test_data['Utest']

# Combine for global scaling
Y_all = np.vstack((Y_train, Y_test))
U_all = np.vstack((U_train, U_test))


In [116]:
# Fit new scalers on full dataset
scaler = StandardScaler().fit(Y_all)
scalerU = StandardScaler().fit(U_all)
x_scaled = scaler.transform(x_all)
u_scaled = scalerU.transform(u_all)

# Save scalers if needed
joblib.dump(scaler, 'data/scaler_all.pkl')
joblib.dump(scalerU, 'data/scalerU_all.pkl')

['data/scalerU_all.pkl']

In [117]:
# Apply scaling to original sets
train_sim = {
    'Y': scaler.transform(Y_train),
    'U': scalerU.transform(U_train)
}

test_sim = {
    'Y': scaler.transform(Y_test),
    'U': scalerU.transform(U_test)
}

In [118]:
# Parameters
nx = train_sim['Y'].shape[1]
nu = train_sim['U'].shape[1]
ny = nx
nsteps = 80
bs = 100
nx_koopman = 10#stavy systemu - kolko Z - k comu skonvergujeme
nx_koopman_baseline = 80
# n_hidden = 32
# n_layers = 2
layers = [60,60,60]
layers_baseline = [60,60,60]



In [119]:
# Dataset creation function (remains unchanged)
def get_data(train_sim, test_sim, nx, nu, nsteps, bs, scaler, scalerU):
    dev_sim = train_sim

    # Combine training and test data for reshaping
    trainY = np.vstack((train_sim['Y'], test_sim['Y']))
    trainU = np.vstack((train_sim['U'], test_sim['U']))

    nsim = trainY.shape[0]
    nbatch = nsim // nsteps
    length = nbatch * nsteps

    trainX = trainY[:length].reshape(nbatch, nsteps, nx)
    trainU = trainU[:length].reshape(nbatch, nsteps, nu)

    train_dataset = DictDataset({
        'Y': torch.tensor(trainX, dtype=torch.float32),
        'Y0': torch.tensor(trainX[:, 0:1, :], dtype=torch.float32),
        'U': torch.tensor(trainU, dtype=torch.float32)
    }, name='train')

    train_loader = DataLoader(train_dataset, batch_size=bs,
                              collate_fn=train_dataset.collate_fn, shuffle=True)

    # Dev set
    nsim = dev_sim['Y'].shape[0]
    nbatch = nsim // nsteps
    length = nbatch * nsteps

    devX = dev_sim['Y'][:length].reshape(nbatch, nsteps, nx)
    devU = dev_sim['U'][:length].reshape(nbatch, nsteps, nu)

    dev_dataset = DictDataset({
        'Y': torch.tensor(devX, dtype=torch.float32),
        'Y0': torch.tensor(devX[:, 0:1, :], dtype=torch.float32),
        'U': torch.tensor(devU, dtype=torch.float32)
    }, name='dev')

    dev_loader = DataLoader(dev_dataset, batch_size=bs,
                            collate_fn=dev_dataset.collate_fn, shuffle=True)

    # Test set (single trajectory)
    nsim = test_sim['Y'].shape[0]
    nbatch = nsim // nsteps
    length = nbatch * nsteps

    testX = test_sim['Y'][:length].reshape(1, nbatch * nsteps, nx)
    testU = test_sim['U'][:length].reshape(1, nbatch * nsteps, nu)

    test_data = {
        'Y': torch.tensor(testX, dtype=torch.float32),
        'Y0': torch.tensor(testX[:, 0:1, :], dtype=torch.float32),
        'U': torch.tensor(testU, dtype=torch.float32)
    }

    return train_loader, dev_loader, test_data

# Get data
train_loader, dev_loader, test_data = get_data(
    train_sim, test_sim, nx, nu, nsteps, bs, scaler, scalerU
)

print("Scaler X Mean:", scaler.mean_)
print("Scaler X Std:", scaler.scale_)
print("Scaler U Mean:", scalerU.mean_)
print("Scaler U Std:", scalerU.scale_)

Scaler X Mean: [59.39946643]
Scaler X Std: [8.2872074]
Scaler U Mean: [57.61569225]
Scaler U Std: [26.34877501]


In [120]:
#train_sim = data
#test_data = data

nsim = train_sim['Y'].shape[0]   # number of simulation steps in the dataset
nsteps = 80   # number of prediction horizon steps in the loss function
bs = 100      # minibatching batch size

#skalovanie

scaler = StandardScaler()
scaler.fit(train_sim['Y'])  # Fit only on training data
joblib.dump(scaler, 'data/scaler.pkl')

scalerU = StandardScaler()
scalerU.fit(train_sim['U'])
joblib.dump(scalerU, 'data/scalerU.pkl')

nx = train_sim['Y'].shape[1]
ny = train_sim['Y'].shape[1]
nu = train_sim['U'].shape[1]

train_loader, dev_loader, test_data = get_data(
    train_sim, test_sim, nx, nu, nsteps, bs, scaler, scalerU
)

KOOPMAN

In [121]:
layers = [60,60,60]
layers_baseline = [60,60,60]

In [122]:
# instantiate output encoder neural net f_y
f_y = blocks.MLP( #g(x)
    ny,
    nx_koopman,
    bias=True,
    linear_map=torch.nn.Linear,
    nonlin=torch.nn.ReLU,
    hsizes=layers,
)
# initial condition encoder
encode_Y0 = Node(f_y, ['Y0'], ['x'], name='encoder_Y0') #nn
# observed trajectory encoder
encode_Y = Node(f_y, ['Y'], ['x_latent'], name='encoder_Y')

#x,x_latenet

In [123]:
# instantiate input encoder net f_u

f_u = torch.nn.Linear(nu, nx_koopman, bias=False)
# initial condition encoder
encode_U = Node(f_u, ['U'], ['u_latent'], name='encoder_U')

In [124]:
# instantiate state decoder neural net f_y_inv
f_y_inv = torch.nn.Linear(nx_koopman, ny, bias=False)
# predicted trajectory decoder
decode_y = Node(f_y_inv, ['x'], ['yhat'], name='decoder_y')

In [125]:
f_y_baseline = blocks.MLP(
    ny,
    nx_koopman_baseline,
    bias=True,
    linear_map=torch.nn.Linear,
    nonlin=torch.nn.ReLU,
    hsizes=layers_baseline,
)
# initial condition encoder
encode_Y0_baseline = Node(f_y_baseline, ['Y0'], ['x'], name='encoder_Y0')
# observed trajectory encoder
encode_Y_baseline = Node(f_y_baseline, ['Y'], ['x_latent'], name='encoder_Y')

f_u_baseline = torch.nn.Linear(nu, nx_koopman_baseline, bias=False)
# initial condition encoder
encode_U_baseline = Node(f_u_baseline, ['U'], ['u_latent'], name='encoder_U')
# instantiate state decoder neural net f_y_inv
f_y_inv_baseline = blocks.MLP(nx_koopman_baseline, ny, bias=True,
                linear_map=torch.nn.Linear,
                nonlin=torch.nn.ELU,
                hsizes=layers_baseline)
# f_y_inv = torch.nn.Linear(nx_koopman, ny, bias=False)
# predicted trajectory decoder
decode_y_baseline = Node(f_y_inv_baseline, ['x'], ['yhat'], name='decoder_y')

In [126]:
# instantiate Koopman operator matrix
stable = False     # if True then provably stable Koopman operator
if stable:
    # SVD factorized Koopman operator with bounded eigenvalues: sigma_min <= \lambda_i <= sigma_max
    K = slim.linear.SVDLinear(nx_koopman, nx_koopman,
                          sigma_min=0.01, sigma_max=1.0, bias=False)
    # SVD penalty variable
    K_reg_error = variable(K.reg_error())
    # SVD penalty loss term
    K_reg_loss = 1.*(K_reg_error == 0.0)
    K_reg_loss.name = 'SVD_loss'
else:
    # linear Koopman operator without guaranteed stability
    K = torch.nn.Linear(nx_koopman, nx_koopman, bias=False)
    K_baseline = torch.nn.Linear(nx_koopman_baseline, nx_koopman_baseline, bias=False)

In [127]:
class Koopman_control(nn.Module):
    """
    Baseline class for Koopman control model
    Implements discrete-time dynamical system:
        x_k+1 = K x_k + u_k
    with variables:
        x_k - latent states
        u_k - latent control inputs
    """

    def __init__(self, K):
        super().__init__()
        self.K = K

    def forward(self, x, u):
        """
        :param x: (torch.Tensor, shape=[batchsize, nx])
        :param u: (torch.Tensor, shape=[batchsize, nx])
        :return: (torch.Tensor, shape=[batchsize, nx])
        """
        x = self.K(x) + u
        return x

In [128]:
# symbolic Koopman model with control inputs
Koopman = Node(Koopman_control(K), ['x', 'u_latent'], ['x'], name='K')
Koopman_baseline = Node(Koopman_control(K_baseline), ['x', 'u_latent'], ['x'], name='K')

# latent Koopmann rollout
dynamics_model = System([Koopman], name='Koopman', nsteps=nsteps)
dynamics_model_baseline = System([Koopman_baseline], name='Koopman', nsteps=nsteps)

In [129]:
# put all nodes of the Koopman model together in a list of nodes
nodes = [encode_Y0, encode_Y, encode_U, dynamics_model, decode_y]

nodes_baseline = [encode_Y0_baseline, encode_Y_baseline, encode_U_baseline, dynamics_model_baseline, decode_y_baseline]

In [130]:
# variables
Y = variable("Y")  # observed
yhat = variable('yhat')  # predicted output
x_latent = variable('x_latent')  # encoded output trajectory in the latent space
u_latent = variable('u_latent')  # encoded input trajectory in the latent space
x = variable('x')  # Koopman latent space trajectory

xu_latent = x_latent + u_latent  # latent state trajectory

# output trajectory tracking loss
y_loss = 10. * (yhat[:, 1:-1, :] == Y[:, 1:, :]) ^ 2
y_loss.name = "y_loss"

# one-step tracking loss
onestep_loss = 1.*(yhat[:, 1, :] == Y[:, 1, :])^2
onestep_loss.name = "onestep_loss"

# reconstruction loss
reconstruction_loss = 20.*(yhat[:, 0, :] == Y[:, 0, :])^2
reconstruction_loss.name = "reconstruction_loss"


# latent trajectory tracking loss
x_loss = 1. * (x[:, 1:-1, :] == xu_latent[:, 1:, :]) ^ 2
x_loss.name = "x_loss"


In [131]:
# aggregate list of objective terms and constraints
objectives = [y_loss, x_loss, onestep_loss, reconstruction_loss]

if stable:
    objectives.append(K_reg_loss)

# create constrained optimization loss
loss = PenaltyLoss(objectives, constraints=[])

# construct constrained optimization problem
problem = Problem(nodes, loss)

problem_baseline = Problem(nodes_baseline, loss)

In [132]:
print(problem.input_keys)
print(problem.output_keys)

['x', 'Y0', 'x_latent', 'yhat', 'u_latent', 'U', 'Y']
['C_ineq_violations', 'objective_loss', 'x', 'C_values', 'C_ineq_values', 'x_latent', 'yhat', 'penalty_loss', 'C_eq_violations', 'u_latent', 'loss', 'C_eq_values', 'C_violations']


In [133]:
optimizer = torch.optim.Adam(problem.parameters(), lr=0.001)

trainer = Trainer(
    problem,
    train_loader,
    dev_loader,
    test_data,
    optimizer,
    patience=200,
    warmup=100,
    epochs=2000,
    eval_metric="dev_loss",
    train_metric="train_loss",
    dev_metric="dev_loss",
    test_metric="dev_loss",
)


In [134]:
# %% train
start = tim.time()
best_model = trainer.train()
problem.load_state_dict(best_model)
end = tim.time()


epoch: 0  train_loss: 30.54664421081543
epoch: 1  train_loss: 29.69089698791504
epoch: 2  train_loss: 28.880287170410156
epoch: 3  train_loss: 28.087902069091797
epoch: 4  train_loss: 27.31137466430664
epoch: 5  train_loss: 26.55260467529297
epoch: 6  train_loss: 25.79216766357422
epoch: 7  train_loss: 25.015832901000977
epoch: 8  train_loss: 24.2136173248291
epoch: 9  train_loss: 23.387624740600586
epoch: 10  train_loss: 22.538957595825195
epoch: 11  train_loss: 21.655426025390625
epoch: 12  train_loss: 20.737083435058594
epoch: 13  train_loss: 19.777828216552734
epoch: 14  train_loss: 18.776077270507812
epoch: 15  train_loss: 17.734783172607422
epoch: 16  train_loss: 16.659259796142578
epoch: 17  train_loss: 15.557523727416992
epoch: 18  train_loss: 14.436655044555664
epoch: 19  train_loss: 13.316963195800781
epoch: 20  train_loss: 12.209177017211914
epoch: 21  train_loss: 11.118131637573242
epoch: 22  train_loss: 10.070685386657715
epoch: 23  train_loss: 9.08350944519043
epoch: 24  

In [135]:
print("yhat shape:", test_outputs['yhat'].shape)
print("Y shape:", test_data['Y'].shape)

NameError: name 'test_outputs' is not defined

In [ ]:
problem.nodes[3].nsteps = 5200

In [ ]:
# Test set results



start = tim.time()
test_outputs = problem.step(test_data)
end = tim.time()
print(f"Elapsed time test: {end-start:.2f} sec")

pred_traj = test_outputs['yhat'][:, 1:-1, :].detach().numpy().reshape(-1, nx).T
true_traj = test_data['Y'][:, 1:pred_traj.shape[1]+1, ].detach().numpy().reshape(-1, nx).T
input_traj = test_data['U'].detach().numpy().reshape(-1, nu).T

In [ ]:
problem_baseline.nodes[3].nsteps = 2000

start = tim.time()
test_outputs = problem_baseline.step(test_data)
end = tim.time()
print(f"Elapsed time test: {end-start:.2f} sec")

baseline_y = test_outputs['yhat'][:, 1:-1, :].detach().numpy().reshape(-1, nx).T

STREJC

In [ ]:
from scipy.signal import cont2discrete

# Given parameters
gain = 1.3707   # Example gain
tau = 43# Example time constant
T = 1    # Sampling time (choose based on your application)

# Continuous-time state-space matrices
A = np.array([[-1/tau]])
B = np.array([[gain/tau]])
C = np.array([[1]])
D = np.array([[0]])

# Discretize using cont2discrete
system = (A, B, C, D)
discrete_system = cont2discrete(system, T, method='zoh')

# Extract discrete-time matrices
A_d, B_d, C_d, D_d, _ = discrete_system

print("Discrete A matrix:", A_d)
print("Discrete B matrix:", B_d)
print("Discrete C matrix:", C_d)
print("Discrete D matrix:", D_d)


In [ ]:
x0 = test_data['Y0'][0].detach().numpy()
input_traj = test_data['U'].detach().numpy()[0]
y_strejc = np.zeros((nx, 2001))
y_strejc[:, 0] = x0
for i in range(2000):
    y_strejc[:,i+1] = A_d @ y_strejc[:,i] + B_d @ input_traj[i]
    #if i%50 == 0 and i<3599:
    #    y_strejc[:,i+1] = true_traj[:,i]
y_strejc = y_strejc.T

In [ ]:
# plot trajectories
figsize = 25
fig, ax = plt.subplots(nx + nu, figsize=(figsize, figsize))
part = 2000

x_labels = [f'$y_{k}$' for k in range(len(true_traj))]
for row, (t1, t2, t3, t4, label) in enumerate(zip(true_traj, pred_traj, y_strejc.T, baseline_y, x_labels)):
    axe = ax[row]
    axe.set_ylabel(label, rotation=0, labelpad=20, fontsize=figsize)
    axe.plot(t1[:part], 'c', linewidth=4.0, label='True')
    axe.plot(t2[:part], 'm--', linewidth=4.0, label='Pred')
    axe.plot(t3[:part], 'r--', linewidth=4.0, label='Strejc')
    axe.tick_params(labelbottom=False, labelsize=figsize)
axe.tick_params(labelbottom=True, labelsize=figsize)
axe.legend(fontsize=figsize)

u_labels = [f'$u_{k}$' for k in range(1)]
for row, (u, label) in enumerate(zip(input_traj, u_labels)):
    axe = ax[1]# ax[row+nx]
    axe.plot(u[:part].T, linewidth=4.0, label='inputs')
    axe.legend(fontsize=figsize)
    axe.set_ylabel(label, rotation=0, labelpad=20, fontsize=figsize)
    axe.tick_params(labelbottom=True, labelsize=figsize)

ax[-1].set_xlabel('$time$', fontsize=figsize)
plt.tight_layout()

In [ ]:
A = K.weight.detach().numpy()
B = f_u.weight.detach().numpy()
C = f_y_inv.weight.detach().numpy()
D = np.zeros((ny, nu))

np.save('./data/A_wC_all.npy', A)
np.save('./data/B_wC_all.npy', B)
np.save('./data/C_wC_all.npy', C)

In [ ]:
def controllability_test(A, B):
    """
    Test the controllability of a system given state-space matrices A and B.

    Parameters:
    A (ndarray): State matrix of size (n, n)
    B (ndarray): Input matrix of size (n, m)

    Returns:
    bool: True if the system is controllable, False otherwise.
    """
    n = A.shape[0]  # Number of states
    controllability_matrix = B
    
    # Compute [B, AB, A^2B, ..., A^(n-1)B]
    for i in range(1, n):
        controllability_matrix = np.hstack((controllability_matrix, np.linalg.matrix_power(A, i) @ B))
    
    # Check rank of the controllability matrix
    rank = np.linalg.matrix_rank(controllability_matrix)
    return rank == n

is_controllable = controllability_test(A, B)
print(f"System is controllable: {is_controllable}")

def analyze_controllability(A, B):
    n = A.shape[0]
    controllability_matrix = B
    for i in range(1, n):
        controllability_matrix = np.hstack((controllability_matrix, np.linalg.matrix_power(A, i) @ B))
    
    rank = np.linalg.matrix_rank(controllability_matrix)
    return controllability_matrix, rank

C_matrix, rank = analyze_controllability(A, B)
print("Controllability Matrix:")
print(C_matrix)
print(f"Rank: {rank}/{A.shape[0]}")



In [ ]:
Qy = np.array([[10]]) #scaler.transform(20+ys)# np.array([[20]])  # Quadratic term
Qu = np.array([[1]]) #scalerU.transform(1+us)# np.array([[1]])  # Quadratic term
N = 40  # Horizon length
nu = Qu.shape[0]  # Number of inputs

umax = scalerU.transform([[100]])
umin = scalerU.transform([[20]])
ymax = scaler.transform([[70]]).reshape(1,-1).T
ymin = scaler.transform([[20]]).reshape(1,-1).T
sim_length = 200

nx = A.shape[0]
nu = B.shape[1]
y = np.zeros((ny, sim_length+1))
u = np.zeros((nu, sim_length))
init_cond = np.array([[-2]])
x0 = get_x(init_cond)
x[:, 0] = x0.flatten()
y[:, 0] = init_cond.flatten()

x_basline = np.zeros((nx_koopman_baseline, sim_length+1))
x0_baseline = get_x_baseline(init_cond)
x_basline[:,0] = x0_baseline.flatten()

In [ ]:
nu = Qu.shape[0]  # Number of inputs

umax = scalerU.transform([[100]])
umin = scalerU.transform([[20]])
ymax = scaler.transform([[70]]).reshape(1,-1).T
ymin = scaler.transform([[20]]).reshape(1,-1).T
#sim_length = 100


nx = A.shape[0]
nu = B.shape[1]
y = np.zeros((ny, sim_length+1))
u = np.zeros((nu, sim_length))
x0 = get_x(init_cond)
x[:, 0] = x0.flatten()
y[:, 0] = init_cond.flatten()

x_basline = np.zeros((nx_koopman_baseline, sim_length+1))
x0_baseline = get_x_baseline(init_cond)
x_basline[:,0] = x0_baseline.flatten()

In [ ]:
start = tim.time()
for i in range(sim_length):
    Q = dense.quad_form_or(A_d, B_d, C_d, D_d, Qy, Qu, N)                      
    c = dense.lin_form_or(A_d, B_d, C_d, D_d, Qy, N, y[:,i])    
    F = dense.constraint_matrix_or(A_d,B_d, C_d, D_d, N)                   
    g = dense.upper_bound_or(A_d, C_d, N, y[:,i].reshape(-1,1), ymax, ymin, umax, umin)
    
    U = cp.Variable((N, nu))
    
    objective = cp.Minimize(cp.quad_form(U, Q) + c @ U)
    constraints = [F @ U <= g]
    
    problem_mpc = cp.Problem(objective, constraints)
    problem_mpc.solve()
    
    u[:, i] = U.value[0, :].reshape(-1, 1)
    # uin = scalerU.transform(u[:, i]+us)
    
    # propagation in time
    y[:,i+1], x_basline[:,i+1] = y_plus_baseline(x_basline[:,i], np.array([-2])) #u[:, i]
    
    noise = np.random.normal(noise_mean, noise_std)
    y[:,i+1] = y[:,i+1] #+ noise
    #y[:,i+1] = scaler.inverse_transform(y[:,i+1].reshape(1,-1)).flatten()
    
    
    x0 = get_x(y[:, i+1].reshape(1,-1))
    #print(i)
    
end = tim.time()
print(f"Elapsed time MPC: {end-start:.2f} sec")

In [ ]:
u_strejc = scalerU.inverse_transform(u)[0]
plt.figure()
plt.plot(u_strejc, label='u_strejc')
plt.plot(u_koopman, label='u_koopman')
plt
plt.legend()
plt.show()

In [ ]:
y_strejc = scaler.inverse_transform(y)
plt.figure()
plt.plot(y_strejc[0, :], label='y_strejc')
plt.plot(y_koopman[0,:], label='y_koopman')
plt.axhline(y=ys, color='r', linestyle='--', label='T setpoint')
plt.legend()
plt.show()

In [ ]:
J_koopman = 0
J_strejc = 0
u_ktest = scalerU.transform(u_koopman.reshape(-1,1))
u_stest = scalerU.transform(u_strejc.reshape(-1,1))
y_ktest = scaler.transform(y_koopman.T)
y_stest = scaler.transform(y_strejc.T)
for i in range(sim_length):
    J_koopman += Qy*y_ktest[i,0]**2 + Qu*u_ktest[i]**2
    J_strejc += Qy*y_stest[i,0]**2 + Qu*u_stest[i]**2

In [ ]:
# After training Koopman:
A = K.weight.detach().numpy()
B = f_u.weight.detach().numpy()
C = f_y_inv.weight.detach().numpy()  # Use decoder weights for Koopman
D = np.zeros((C.shape[0], B.shape[1]))

np.save('./data/A_wC_all.npy', A)
np.save('./data/B_wC_all.npy', B)
np.save('./data/C_wC_all.npy', C)
np.save('./data/D_wC_all.npy', D)
